# Gold Layer Diagnostics

Sanity-checks all Gold tables and stores one row per check in:

`data_lakehouse_databricks.gold.diagnostic_gold`

Checks include:

- table existence
- duplicate business keys
- exact duplicate rows
- missing values in required columns
- expected nullable columns
- invalid date ranges
- invalid numeric measures
- customer/product referential integrity
- product validity-period overlap
- fact rows that do not match the correct historical product version

`gold_dim_product.end_date` is treated as intentionally nullable because the latest/current product version has no end date.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType
)
import json


## Configuration


In [0]:
CATALOG = "data_lakehouse_databricks"
GOLD_SCHEMA = "gold"

DIAGNOSTIC_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.diagnostic_gold"

TABLE_CONFIG = {
    "gold_dim_customers": {
        "keys": ["customer_id"],
        "secondary_unique_keys": [["customer_key"]],
        "required_columns": [
            "customer_id",
            "customer_key",
            "firstname",
            "lastname",
            "marital_status",
            "gender",
            "country",
            "birth_date",
            "creation_date",
        ],
        "nullable_columns": [],
    },

    "gold_dim_product": {
        "keys": ["product_id"],
        "secondary_unique_keys": [["product_key", "start_date"]],
        "required_columns": [
            "product_id",
            "product_key",
            "product_name",
            "cost",
            "product_line",
            "start_date",
            "category_id",
            "category",
            "subcategory",
            "maintenance",
        ],
        # NULL end_date is meaningful for the current product version.
        "nullable_columns": ["end_date"],
    },

    "gold_fact_sales": {
        # This matches the grain used in the Silver cleaning pipeline.
        "keys": ["order_number", "product_key"],
        "secondary_unique_keys": [],
        "required_columns": [
            "order_number",
            "product_key",
            "customer_id",
            "order_date",
            "ship_date",
            "due_date",
            "sales_amount",
            "quantity",
            "price",
        ],
        "nullable_columns": [],
    },
}


## Generic diagnostic helpers


In [0]:
RESULT_SCHEMA = StructType([
    StructField("table_name", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("check_type", StringType(), False),
    StructField("columns_checked", StringType(), True),
    StructField("total_rows", LongType(), False),
    StructField("failed_rows", LongType(), False),
    StructField("pass_percentage", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("details", StringType(), True),
])


def table_exists(full_name):
    try:
        spark.table(full_name)
        return True
    except Exception:
        return False


def make_result(
    table_name,
    check_name,
    check_type,
    total_rows,
    failed_rows,
    columns_checked=None,
    details=None,
    warn_only=False,
):
    total_rows = int(total_rows)
    failed_rows = int(failed_rows)

    if total_rows == 0:
        pass_percentage = 100.0 if failed_rows == 0 else 0.0
    else:
        pass_percentage = round(
            ((total_rows - failed_rows) / total_rows) * 100.0,
            3
        )

    if failed_rows == 0:
        status = "PASS"
    elif warn_only:
        status = "WARN"
    else:
        status = "FAIL"

    return {
        "table_name": table_name,
        "check_name": check_name,
        "check_type": check_type,
        "columns_checked": (
            json.dumps(columns_checked)
            if columns_checked is not None
            else None
        ),
        "total_rows": total_rows,
        "failed_rows": failed_rows,
        "pass_percentage": pass_percentage,
        "status": status,
        "details": details,
    }


def count_duplicate_rows(df, keys):
    if not keys:
        return 0

    dup_groups = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
    )

    row = dup_groups.agg(
        F.coalesce(
            F.sum(F.col("count") - F.lit(1)),
            F.lit(0)
        ).alias("duplicate_rows")
    ).first()

    return int(row["duplicate_rows"])


def count_exact_duplicates(df):
    total = df.count()
    distinct = df.distinct().count()
    return total - distinct


def count_nulls(df, column_name):
    return (
        df.filter(F.col(column_name).isNull())
          .count()
    )


## Table-level checks


In [0]:
results = []

for table_name, config in TABLE_CONFIG.items():

    full_name = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"

    print("=" * 90)
    print(f"Checking {full_name}")
    print("=" * 90)

    if not table_exists(full_name):
        results.append(
            make_result(
                table_name=table_name,
                check_name="table_exists",
                check_type="existence",
                total_rows=1,
                failed_rows=1,
                details=f"Table not found: {full_name}",
            )
        )
        continue

    df = spark.table(full_name)
    total_rows = df.count()

    # ---------------------------------------------------------
    # 1. Table existence
    # ---------------------------------------------------------
    results.append(
        make_result(
            table_name=table_name,
            check_name="table_exists",
            check_type="existence",
            total_rows=1,
            failed_rows=0,
            details=f"Table exists with {total_rows} rows.",
        )
    )

    # ---------------------------------------------------------
    # 2. Empty table
    # ---------------------------------------------------------
    results.append(
        make_result(
            table_name=table_name,
            check_name="table_not_empty",
            check_type="volume",
            total_rows=1,
            failed_rows=1 if total_rows == 0 else 0,
            details=f"Row count = {total_rows}.",
        )
    )

    # ---------------------------------------------------------
    # 3. Exact duplicate rows
    # ---------------------------------------------------------
    exact_duplicates = count_exact_duplicates(df)

    results.append(
        make_result(
            table_name=table_name,
            check_name="no_exact_duplicate_rows",
            check_type="duplicates",
            total_rows=total_rows,
            failed_rows=exact_duplicates,
            columns_checked=df.columns,
            details=(
                f"{exact_duplicates} exact duplicate rows found."
            ),
        )
    )

    # ---------------------------------------------------------
    # 4. Primary/business key duplicates
    # ---------------------------------------------------------
    keys = config["keys"]

    missing_key_columns = [
        col_name
        for col_name in keys
        if col_name not in df.columns
    ]

    if missing_key_columns:
        results.append(
            make_result(
                table_name=table_name,
                check_name="primary_key_columns_exist",
                check_type="schema",
                total_rows=1,
                failed_rows=1,
                columns_checked=keys,
                details=(
                    "Missing configured key columns: "
                    + ", ".join(missing_key_columns)
                ),
            )
        )
    else:
        duplicate_rows = count_duplicate_rows(df, keys)

        results.append(
            make_result(
                table_name=table_name,
                check_name="primary_key_unique",
                check_type="duplicates",
                total_rows=total_rows,
                failed_rows=duplicate_rows,
                columns_checked=keys,
                details=(
                    f"{duplicate_rows} rows exceed the configured "
                    f"business-key grain {keys}."
                ),
            )
        )

    # ---------------------------------------------------------
    # 5. Additional uniqueness checks
    # ---------------------------------------------------------
    for unique_keys in config.get("secondary_unique_keys", []):
        missing = [
            col_name
            for col_name in unique_keys
            if col_name not in df.columns
        ]

        if missing:
            results.append(
                make_result(
                    table_name=table_name,
                    check_name=(
                        "unique_" + "_".join(unique_keys)
                    ),
                    check_type="schema",
                    total_rows=1,
                    failed_rows=1,
                    columns_checked=unique_keys,
                    details=(
                        "Missing configured columns: "
                        + ", ".join(missing)
                    ),
                )
            )
            continue

        duplicate_rows = count_duplicate_rows(
            df,
            unique_keys
        )

        results.append(
            make_result(
                table_name=table_name,
                check_name=(
                    "unique_" + "_".join(unique_keys)
                ),
                check_type="duplicates",
                total_rows=total_rows,
                failed_rows=duplicate_rows,
                columns_checked=unique_keys,
                details=(
                    f"{duplicate_rows} duplicate rows for "
                    f"{unique_keys}."
                ),
            )
        )

    # ---------------------------------------------------------
    # 6. Required-column null checks
    # ---------------------------------------------------------
    for column_name in config["required_columns"]:

        if column_name not in df.columns:
            results.append(
                make_result(
                    table_name=table_name,
                    check_name=f"not_null_{column_name}",
                    check_type="schema",
                    total_rows=1,
                    failed_rows=1,
                    columns_checked=[column_name],
                    details=(
                        f"Required column {column_name} "
                        "does not exist."
                    ),
                )
            )
            continue

        null_count = count_nulls(df, column_name)

        results.append(
            make_result(
                table_name=table_name,
                check_name=f"not_null_{column_name}",
                check_type="missing_values",
                total_rows=total_rows,
                failed_rows=null_count,
                columns_checked=[column_name],
                details=(
                    f"{null_count} NULL values in "
                    f"{column_name}."
                ),
            )
        )

    # ---------------------------------------------------------
    # 7. Expected nullable columns
    # ---------------------------------------------------------
    for column_name in config.get(
        "nullable_columns", []
    ):
        if column_name not in df.columns:
            results.append(
                make_result(
                    table_name=table_name,
                    check_name=f"nullable_{column_name}",
                    check_type="schema",
                    total_rows=1,
                    failed_rows=1,
                    columns_checked=[column_name],
                    details=f"Column {column_name} is missing.",
                )
            )
            continue

        null_count = count_nulls(df, column_name)

        # Informational/WARN rather than FAIL because NULL is valid here.
        results.append(
            make_result(
                table_name=table_name,
                check_name=f"expected_nullable_{column_name}",
                check_type="missing_values_expected",
                total_rows=total_rows,
                failed_rows=null_count,
                columns_checked=[column_name],
                details=(
                    f"{null_count} NULL values. NULL is "
                    "allowed for this column by design."
                ),
                warn_only=True,
            )
        )


## Gold-specific business-rule checks


In [0]:
# =============================================================
# CUSTOMER DIMENSION
# =============================================================

customer_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_dim_customers"

if table_exists(customer_table):
    customers = spark.table(customer_table)
    n = customers.count()

    # birth_date should not be in the future
    if "birth_date" in customers.columns:
        bad = (
            customers
            .filter(
                F.col("birth_date").isNotNull()
                & (F.col("birth_date") > F.current_date())
            )
            .count()
        )

        results.append(
            make_result(
                "gold_dim_customers",
                "birth_date_not_in_future",
                "business_rule",
                n,
                bad,
                ["birth_date"],
                f"{bad} customers have future birth dates.",
            )
        )

    # creation date should not be in the future
    if "creation_date" in customers.columns:
        bad = (
            customers
            .filter(
                F.col("creation_date").isNotNull()
                & (F.col("creation_date") > F.current_date())
            )
            .count()
        )

        results.append(
            make_result(
                "gold_dim_customers",
                "creation_date_not_in_future",
                "business_rule",
                n,
                bad,
                ["creation_date"],
                f"{bad} customers have future creation dates.",
            )
        )


# =============================================================
# PRODUCT DIMENSION
# =============================================================

product_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_dim_product"

if table_exists(product_table):
    products = spark.table(product_table)
    n = products.count()

    # Cost should not be negative.
    if "cost" in products.columns:
        bad = (
            products
            .filter(
                F.col("cost").isNotNull()
                & (F.col("cost") < 0)
            )
            .count()
        )

        results.append(
            make_result(
                "gold_dim_product",
                "cost_non_negative",
                "business_rule",
                n,
                bad,
                ["cost"],
                f"{bad} products have negative cost.",
            )
        )

    # end_date must be >= start_date when end_date is present.
    if {"start_date", "end_date"}.issubset(products.columns):
        bad = (
            products
            .filter(
                F.col("start_date").isNotNull()
                & F.col("end_date").isNotNull()
                & (F.col("end_date") < F.col("start_date"))
            )
            .count()
        )

        results.append(
            make_result(
                "gold_dim_product",
                "valid_product_date_range",
                "business_rule",
                n,
                bad,
                ["start_date", "end_date"],
                (
                    f"{bad} product versions have "
                    "end_date < start_date."
                ),
            )
        )

        # Detect overlapping validity periods for the same product_key.
        w = Window.partitionBy("product_key").orderBy(
            F.col("start_date").asc_nulls_last()
        )

        product_ranges = (
            products
            .withColumn(
                "_previous_end_date",
                F.lag("end_date").over(w)
            )
        )

        overlaps = (
            product_ranges
            .filter(
                F.col("_previous_end_date").isNotNull()
                & F.col("start_date").isNotNull()
                & (
                    F.col("start_date")
                    <= F.col("_previous_end_date")
                )
            )
            .count()
        )

        results.append(
            make_result(
                "gold_dim_product",
                "no_overlapping_product_versions",
                "business_rule",
                n,
                overlaps,
                ["product_key", "start_date", "end_date"],
                (
                    f"{overlaps} product versions overlap "
                    "a previous validity interval."
                ),
            )
        )

        # At most one current version (end_date IS NULL) per product_key.
        current_version_duplicates = (
            products
            .filter(F.col("end_date").isNull())
            .groupBy("product_key")
            .count()
            .filter(F.col("count") > 1)
        )

        duplicate_current_rows = (
            current_version_duplicates
            .agg(
                F.coalesce(
                    F.sum(F.col("count") - 1),
                    F.lit(0)
                ).alias("bad")
            )
            .first()["bad"]
        )

        duplicate_current_rows = int(
            duplicate_current_rows or 0
        )

        results.append(
            make_result(
                "gold_dim_product",
                "one_current_version_per_product",
                "business_rule",
                n,
                duplicate_current_rows,
                ["product_key", "end_date"],
                (
                    f"{duplicate_current_rows} extra current "
                    "product versions found."
                ),
            )
        )


# =============================================================
# SALES FACT
# =============================================================

sales_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_fact_sales"

if table_exists(sales_table):
    sales = spark.table(sales_table)
    n = sales.count()

    # Positive/non-negative measures
    numeric_rules = {
        "quantity": (">", 0),
        "price": (">=", 0),
        "sales_amount": (">=", 0),
    }

    for column_name, (operator, threshold) in numeric_rules.items():
        if column_name not in sales.columns:
            continue

        if operator == ">":
            bad_df = sales.filter(
                F.col(column_name).isNotNull()
                & (F.col(column_name) <= threshold)
            )
        else:
            bad_df = sales.filter(
                F.col(column_name).isNotNull()
                & (F.col(column_name) < threshold)
            )

        bad = bad_df.count()

        results.append(
            make_result(
                "gold_fact_sales",
                f"{column_name}_valid",
                "business_rule",
                n,
                bad,
                [column_name],
                (
                    f"{bad} rows violate "
                    f"{column_name} {operator} {threshold}."
                ),
            )
        )

    # Order / ship / due chronology
    if {"order_date", "ship_date"}.issubset(sales.columns):
        bad = (
            sales
            .filter(
                F.col("order_date").isNotNull()
                & F.col("ship_date").isNotNull()
                & (F.col("ship_date") < F.col("order_date"))
            )
            .count()
        )

        results.append(
            make_result(
                "gold_fact_sales",
                "ship_date_on_or_after_order_date",
                "business_rule",
                n,
                bad,
                ["order_date", "ship_date"],
                f"{bad} sales have ship_date < order_date.",
            )
        )

    if {"order_date", "due_date"}.issubset(sales.columns):
        bad = (
            sales
            .filter(
                F.col("order_date").isNotNull()
                & F.col("due_date").isNotNull()
                & (F.col("due_date") < F.col("order_date"))
            )
            .count()
        )

        results.append(
            make_result(
                "gold_fact_sales",
                "due_date_on_or_after_order_date",
                "business_rule",
                n,
                bad,
                ["order_date", "due_date"],
                f"{bad} sales have due_date < order_date.",
            )
        )


## Referential integrity checks


In [0]:
# -------------------------------------------------------------
# fact_sales.customer_id -> dim_customers.customer_id
# -------------------------------------------------------------

if table_exists(sales_table) and table_exists(customer_table):

    sales = spark.table(sales_table)
    customers = spark.table(customer_table)

    customer_keys = (
        customers
        .select(
            F.col("customer_id").alias("_customer_id")
        )
        .where(F.col("_customer_id").isNotNull())
        .distinct()
    )

    orphan_customers = (
        sales
        .where(F.col("customer_id").isNotNull())
        .join(
            customer_keys,
            sales["customer_id"]
            == customer_keys["_customer_id"],
            "left_anti"
        )
        .count()
    )

    results.append(
        make_result(
            "gold_fact_sales",
            "customer_fk_exists",
            "referential_integrity",
            sales.count(),
            orphan_customers,
            ["customer_id"],
            (
                f"{orphan_customers} sales rows reference "
                "a customer_id absent from gold_dim_customers."
            ),
        )
    )


# -------------------------------------------------------------
# fact_sales.product_key -> dim_product.product_key
# -------------------------------------------------------------

if table_exists(sales_table) and table_exists(product_table):

    sales = spark.table(sales_table)
    products = spark.table(product_table)

    product_keys = (
        products
        .select(
            F.col("product_key").alias("_product_key")
        )
        .where(F.col("_product_key").isNotNull())
        .distinct()
    )

    orphan_products = (
        sales
        .where(F.col("product_key").isNotNull())
        .join(
            product_keys,
            sales["product_key"]
            == product_keys["_product_key"],
            "left_anti"
        )
        .count()
    )

    results.append(
        make_result(
            "gold_fact_sales",
            "product_fk_exists",
            "referential_integrity",
            sales.count(),
            orphan_products,
            ["product_key"],
            (
                f"{orphan_products} sales rows reference "
                "a product_key absent from gold_dim_product."
            ),
        )
    )

    # ---------------------------------------------------------
    # Historical product-version match:
    # a sale should match exactly one product validity period.
    # ---------------------------------------------------------
    sales_with_id = (
        sales
        .withColumn(
            "_sale_row_id",
            F.monotonically_increasing_id()
        )
    )

    matched = (
        sales_with_id.alias("s")
        .join(
            products.alias("p"),
            (
                (F.col("s.product_key") == F.col("p.product_key"))
                & (
                    F.col("s.order_date")
                    >= F.col("p.start_date")
                )
                & (
                    F.col("p.end_date").isNull()
                    | (
                        F.col("s.order_date")
                        <= F.col("p.end_date")
                    )
                )
            ),
            "left"
        )
        .groupBy(
            F.col("s._sale_row_id").alias("_sale_row_id")
        )
        .agg(
            F.count(
                F.col("p.product_id")
            ).alias("_version_matches")
        )
    )

    no_version_match = (
        matched
        .filter(F.col("_version_matches") == 0)
        .count()
    )

    multiple_version_match = (
        matched
        .filter(F.col("_version_matches") > 1)
        .count()
    )

    results.append(
        make_result(
            "gold_fact_sales",
            "exactly_one_historical_product_version",
            "referential_integrity",
            sales.count(),
            no_version_match + multiple_version_match,
            ["product_key", "order_date"],
            (
                f"{no_version_match} sales match no historical "
                f"product version; {multiple_version_match} "
                "match more than one."
            ),
        )
    )


## Save diagnostics


In [0]:
diagnostic_df = (
    spark.createDataFrame(
        results,
        schema=RESULT_SCHEMA
    )
    .orderBy(
        F.when(F.col("status") == "FAIL", 0)
         .when(F.col("status") == "WARN", 1)
         .otherwise(2),
        "table_name",
        "check_type",
        "check_name",
    )
)

display(diagnostic_df)

(
    diagnostic_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIAGNOSTIC_TABLE)
)

print(f"Saved diagnostics to: {DIAGNOSTIC_TABLE}")


## Overall summary


In [0]:
summary_df = (
    diagnostic_df
    .groupBy("table_name", "status")
    .count()
    .orderBy("table_name", "status")
)

display(summary_df)

failed_checks = (
    diagnostic_df
    .filter(F.col("status") == "FAIL")
    .count()
)

warning_checks = (
    diagnostic_df
    .filter(F.col("status") == "WARN")
    .count()
)

print(f"FAILED checks: {failed_checks}")
print(f"WARNING checks: {warning_checks}")

if failed_checks == 0:
    print("Gold sanity checks PASSED: no failing checks.")
else:
    print(
        "Gold sanity checks FAILED. "
        "Inspect data_lakehouse_databricks.gold.diagnostic_gold."
    )
